In [1]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [2]:
Processed_Dataset = "20230930"

In [3]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [4]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [5]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 14915, range: 339)> Size: 40MB
array([[0.43407383, 0.42714838, 0.42144632, ...,        nan,        nan,
               nan],
       [0.43407383, 0.42714838, 0.42144632, ...,        nan,        nan,
               nan],
       [0.43407383, 0.42714838, 0.42144632, ...,        nan,        nan,
               nan],
       ...,
       [0.61547125, 0.60639144, 0.59797072, ...,        nan,        nan,
               nan],
       [0.61547125, 0.60639144, 0.59797072, ...,        nan,        nan,
               nan],
       [0.61547125, 0.60639144, 0.59797072, ...,        nan,        nan,
               nan]], shape=(14915, 339))
Coordinates:
  * Time     (Time) datetime64[ns] 119kB 2023-09-30T20:02:27 ... 2023-10-01T0...
  * range    (range) float32 1kB 111.8 134.2 156.5 ... 1.194e+04 1.197e+04
    TAlt     (range) float32 1kB ...
    HAlt     (range) float32 1kB ...
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    mo

In [6]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")